## Example 1 — the installed version, and torch.Tensor has the DLPack C exchange hook after the import

In [ ]:
import importlib.metadata
import subprocess
import sys

version = importlib.metadata.version("torch_c_dlpack_ext")
# In a child interpreter, so the package is imported fresh and "before the import" is really before it.
# A torch that already has the hook keeps it unchanged; on an older torch the package installs it at import.
check = ("import torch; before = getattr(torch.Tensor, '__dlpack_c_exchange_api__', None); import torch_c_dlpack_ext; "
         "after = getattr(torch.Tensor, '__dlpack_c_exchange_api__', None); "
         "print(type(after).__name__, before is None or before is after)")
child = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True, timeout=600)
assert child.returncode == 0, child.stderr[-2000:]
assert child.stdout.split()[-2:] == ["PyCapsule", "True"], child.stdout
print(f"torch_c_dlpack_ext {version}: torch.Tensor exposes the DLPack C exchange hook after import")

## Example 2 — a tensor handed over through DLPack is the same memory, not a copy

In [ ]:
import numpy as np
import torch
import torch_c_dlpack_ext

original = torch.arange(12, dtype=torch.float32).reshape(3, 4)
through_torch = torch.from_dlpack(original)
assert torch.equal(through_torch, original) and through_torch.data_ptr() == original.data_ptr()
through_numpy = np.from_dlpack(original)
through_numpy[0, 0] = 99.0
assert original[0, 0].item() == 99.0, "numpy saw a copy, not the tensor's own memory"
print("torch -> torch and torch -> numpy share memory:", tuple(through_numpy.shape), through_numpy.dtype)

## Example 3 — the hand-over tvm-ffi makes (what quack's kernels rely on)

In [ ]:
import importlib.util
import torch
import torch_c_dlpack_ext

if importlib.util.find_spec("tvm_ffi") is None:
    print("SKIP: `tvm_ffi` (apache-tvm-ffi) is not installed -> its hand-over is not exercised")
else:
    import tvm_ffi
    original = torch.arange(6, dtype=torch.int64).reshape(2, 3)
    as_tvm = tvm_ffi.from_dlpack(original)
    assert tuple(as_tvm.shape) == (2, 3) and str(as_tvm.dtype) == "int64", (as_tvm.shape, as_tvm.dtype)
    back = torch.from_dlpack(as_tvm)
    assert torch.equal(back, original) and back.data_ptr() == original.data_ptr()
    # A tvm-ffi function called with a torch tensor gets it through the C hook and hands a torch tensor back.
    identity = tvm_ffi.convert(lambda value: value)
    returned = identity(original)
    assert isinstance(returned, torch.Tensor) and returned.data_ptr() == original.data_ptr()
    print("torch -> tvm_ffi -> torch without a copy; a tvm-ffi function returns a torch.Tensor")

## Example 4 — the same hand-over for a tensor on the GPU

In [ ]:
import importlib.util
import torch
import torch_c_dlpack_ext

assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
on_gpu = torch.arange(6, dtype=torch.float32, device="cuda")
back = torch.from_dlpack(on_gpu)
assert back.is_cuda and back.data_ptr() == on_gpu.data_ptr() and torch.equal(back, on_gpu)
if importlib.util.find_spec("tvm_ffi") is None:
    print("SKIP: `tvm_ffi` (apache-tvm-ffi) is not installed -> its hand-over is not exercised")
else:
    import tvm_ffi
    as_tvm = tvm_ffi.from_dlpack(on_gpu)
    assert "cuda" in str(as_tvm.device), as_tvm.device
    assert torch.from_dlpack(as_tvm).data_ptr() == on_gpu.data_ptr()
    print("GPU tensor handed to tvm_ffi on", as_tvm.device, "and back without a copy")